# 1. Bibliotecas


In [113]:
import pandas as pd
import numpy as np
from datetime import date, datetime, time, timedelta, timezone
from matplotlib import pyplot as plt
import seaborn as sns
import glob
import re
import warnings
import math
import statsmodels.tsa.stattools as st
from dateutil import parser
import chardet


# Funções

In [115]:
#Tratar colunas de data e hora (antigo)
def Tratar_data_hora(df, dt_col, hr_col, hr_format='%H%M %Z'): #, dt_format):
    
    #df[dt_col] = df[dt_col].apply(lambda x: datetime.strptime(x, dt_format)) 
    #df[hr_col] = df[hr_col].apply(lambda x: datetime.strptime(x,hr_format).time()) 
    
    df[dt_col] = df[dt_col].apply(lambda x: parser.parse(x).date()) 
    df[hr_col] = df[hr_col].apply(lambda x: parser.parse(x).time() if ':' in x else datetime.strptime(x,hr_format).time()) 

    
    #Dt_Hr =list(map(lambda x, y: datetime.combine(x, y), df['Dia'], df['Hora']))
    Dt_Hr = df.apply(lambda lin: datetime.combine(lin[dt_col], lin[hr_col]), axis = 1)
    df.insert(loc=2, column='Dt_Hr', value=Dt_Hr)
    
    #ts =  list(map(lambda x: datetime.timestamp(x), Dt_Hr))
    ts = df.apply(lambda lin: datetime.timestamp(lin['Dt_Hr']), axis = 1)
    df.insert(loc=3, column='timestamp', value=ts)
    
    df.drop(columns=[dt_col, hr_col], inplace=True)
    
    return df

In [116]:
#Tratar data hora: parser personalizado
def my_date_parser(x, f='%Y/%m/%d %H%M %Z'): 
    if ':' in x:
        return parser.parse(x)
    else:
        return datetime.strptime(x, f)      


In [117]:
def detectar_encoding(arquivo):
    with open(arquivo, 'rb') as f:
        resultado = chardet.detect(f.read())
    return resultado['encoding']

In [118]:
def concat_dfs(lista_arq, skip=0, sep=';', dec=',', verb=False, colunas=None, loc_cols=None):
    df = pd.DataFrame()
    
    for arquivo, arq_ind in zip(lista_arq, range(len(lista_arq))):
        enc = detectar_encoding(arquivo)
        
        data = pd.read_csv(arquivo, sep=sep, skiprows=skip, decimal=dec, encoding=enc, parse_dates= [[0,1]], 
                           date_parser = my_date_parser).dropna(axis=1, how='all')

        # Metadados de localização
        if colunas is not None: data.columns = colunas
            
        if skip>0 and loc_cols is not None:
            #carregar o cabeçalho de metadados
            meta_data = pd.read_csv(arquivo, sep=sep, nrows=skip, encoding=enc, header= None, index_col=0).T

            #formatar dados de localização
            lat = float(re.sub(r'[^-?\d+(\.\d+)?]', '', meta_data[loc_cols[0]][1].replace(',','.')))
            long=float(re.sub(r'[^-?\d+(\.\d+)?]', '',meta_data[loc_cols[1]][1].replace(',','.')))
            alt = float(re.sub(r'[^-?\d+(\.\d+)?]', '',meta_data[loc_cols[2]][1].replace(',','.')))
        
            data.insert(loc=1, column='Lat', value=lat)  
            data.insert(loc=2, column='Long', value=long)
            data.insert(loc=3, column='Alt', value=alt)
        
        if verb:
            print(arq_ind, ': ', arquivo, 'linhas: ', len(data), 'colunas: ', len(data.columns)) 
            print(data.columns)

        df = pd.concat([df, data], ignore_index=True)

    #coluna de timestamp
    ts =  pd.Series(map(datetime.timestamp, df.iloc[:,0]))
    df.insert(loc=1, column='timestamp', value=ts)    
    return df

# VILA MILITAR

In [120]:

# Listar arquivos

warnings.filterwarnings('ignore')


path = 'VILA_MILITAR'

lista_arq = glob.glob(path + "/*.csv")

lista_arq, len(lista_arq)

(['VILA_MILITAR\\INMET_SE_RJ_A621_RIO DE JANEIRO - VILA MILITAR_01-01-2019_A_31-12-2019.CSV',
  'VILA_MILITAR\\INMET_SE_RJ_A621_RIO DE JANEIRO - VILA MILITAR_01-01-2020_A_31-12-2020.CSV',
  'VILA_MILITAR\\INMET_SE_RJ_A621_RIO DE JANEIRO - VILA MILITAR_01-01-2021_A_31-12-2021.CSV',
  'VILA_MILITAR\\INMET_SE_RJ_A621_RIO DE JANEIRO - VILA MILITAR_01-01-2022_A_31-12-2022.CSV',
  'VILA_MILITAR\\INMET_SE_RJ_A621_RIO DE JANEIRO - VILA MILITAR_01-01-2023_A_31-12-2023.CSV',
  'VILA_MILITAR\\INMET_SE_RJ_A621_RIO DE JANEIRO - VILA MILITAR_01-01-2024_A_30-11-2024.CSV',
  'VILA_MILITAR\\INMET_SE_RJ_A621_VILA MILITAR_01-01-2008_A_31-12-2008.CSV',
  'VILA_MILITAR\\INMET_SE_RJ_A621_VILA MILITAR_01-01-2009_A_31-12-2009.CSV',
  'VILA_MILITAR\\INMET_SE_RJ_A621_VILA MILITAR_01-01-2010_A_31-12-2010.CSV',
  'VILA_MILITAR\\INMET_SE_RJ_A621_VILA MILITAR_01-01-2011_A_31-12-2011.CSV',
  'VILA_MILITAR\\INMET_SE_RJ_A621_VILA MILITAR_01-01-2012_A_31-12-2012.CSV',
  'VILA_MILITAR\\INMET_SE_RJ_A621_VILA MILITAR_01-0

In [121]:
# Gerar dataset e padronizar nomes
nomes = ['Dt_Hr', 'Precip', 'Pres_Atm', 'Pres_Atm_max', 'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv', 'Temp_Amb_max', 'Temp_Amb_min', 'Pto_Orv_max',
        'Pto_Orv_min', 'Umidade_max', 'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj', 'Vento_vel']

local = ['LATITUDE:', 'LONGITUDE:', 'ALTITUDE:']

dataset = concat_dfs(lista_arq, skip=8, colunas=nomes, loc_cols = local, verb=True)

0 :  VILA_MILITAR\INMET_SE_RJ_A621_RIO DE JANEIRO - VILA MILITAR_01-01-2019_A_31-12-2019.CSV linhas:  8760 colunas:  21
Index(['Dt_Hr', 'Lat', 'Long', 'Alt', 'Precip', 'Pres_Atm', 'Pres_Atm_max',
       'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv', 'Temp_Amb_max',
       'Temp_Amb_min', 'Pto_Orv_max', 'Pto_Orv_min', 'Umidade_max',
       'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj', 'Vento_vel'],
      dtype='object')
1 :  VILA_MILITAR\INMET_SE_RJ_A621_RIO DE JANEIRO - VILA MILITAR_01-01-2020_A_31-12-2020.CSV linhas:  8784 colunas:  21
Index(['Dt_Hr', 'Lat', 'Long', 'Alt', 'Precip', 'Pres_Atm', 'Pres_Atm_max',
       'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv', 'Temp_Amb_max',
       'Temp_Amb_min', 'Pto_Orv_max', 'Pto_Orv_min', 'Umidade_max',
       'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj', 'Vento_vel'],
      dtype='object')
2 :  VILA_MILITAR\INMET_SE_RJ_A621_RIO DE JANEIRO - VILA MILITAR_01-01-2021_A_31-12-2021.CSV linhas:  8760 colunas:  21
Index(['Dt_Hr', 'Lat', '

In [122]:
dataset.head(2)

,Dt_Hr,timestamp,Lat,Long,Alt,Precip,Pres_Atm,Pres_Atm_max,Pres_Atm_min,Rad,...,Temp_Amb_max,Temp_Amb_min,Pto_Orv_max,Pto_Orv_min,Umidade_max,Umidade_min,Umidade,Vento_dir,Vento_raj,Vento_vel
0,2019-01-01 00:00:00,1.546312e+09,-22.861389,-43.411389,30.43,0.0,1009.8,1009.9,1009.7,NaN,...,23.9,23.6,22.4,22.0,92.0,91.0,91.0,7.0,2.4,0.6
1,2019-01-01 01:00:00,1.546315e+09,-22.861389,-43.411389,30.43,0.0,1010.4,1010.4,1009.8,NaN,...,23.9,23.4,22.5,21.9,92.0,90.0,92.0,122.0,2.7,0.7


In [123]:
dataset['Dt_Hr'].min()

Timestamp('2007-04-13 00:00:00')

In [124]:
dataset['Dt_Hr'].max()

Timestamp('2024-11-30 23:00:00')

In [125]:
caminho_arquivo = "VILA_MILITAR/CONCATENADO/VILA_MILITAR.csv"
dataset.to_csv(caminho_arquivo, index=False, encoding='utf-8')

# FORTE DE COPACABANA

In [127]:
# Listar arquivos

warnings.filterwarnings('ignore')

path = 'FORTE_COPACABANA'

lista_arq = glob.glob(path + "/*.csv")

lista_arq, len(lista_arq)

(['FORTE_COPACABANA\\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2008_A_31-12-2008.CSV',
  'FORTE_COPACABANA\\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2009_A_31-12-2009.CSV',
  'FORTE_COPACABANA\\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2010_A_31-12-2010.CSV',
  'FORTE_COPACABANA\\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2011_A_31-12-2011.CSV',
  'FORTE_COPACABANA\\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2012_A_31-12-2012.CSV',
  'FORTE_COPACABANA\\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2013_A_31-12-2013.CSV',
  'FORTE_COPACABANA\\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2014_A_31-12-2014.CSV',
  'FORTE_COPACABANA\\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2015_A_31-12-2015.CSV',
  'FORTE_COPACABANA\\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2016_A_31-12-2016.CSV',
  'FORTE_COPACABANA\\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2017_A_31-12-2017.CSV',
  'FORTE_COPACABANA\\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2018_A_31-12-2018.CSV',
  'FORTE_COPACABANA\\INMET_SE_RJ

In [128]:
# Gerar dataset e padronizar nomes
nomes = ['Dt_Hr', 'Precip', 'Pres_Atm', 'Pres_Atm_max', 'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv', 'Temp_Amb_max', 'Temp_Amb_min', 'Pto_Orv_max',
        'Pto_Orv_min', 'Umidade_max', 'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj', 'Vento_vel']

local = ['LATITUDE:', 'LONGITUDE:', 'ALTITUDE:']

dataset = concat_dfs(lista_arq, skip=8, colunas=nomes, loc_cols = local, verb=True)

0 :  FORTE_COPACABANA\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2008_A_31-12-2008.CSV linhas:  8784 colunas:  21
Index(['Dt_Hr', 'Lat', 'Long', 'Alt', 'Precip', 'Pres_Atm', 'Pres_Atm_max',
       'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv', 'Temp_Amb_max',
       'Temp_Amb_min', 'Pto_Orv_max', 'Pto_Orv_min', 'Umidade_max',
       'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj', 'Vento_vel'],
      dtype='object')
1 :  FORTE_COPACABANA\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2009_A_31-12-2009.CSV linhas:  8760 colunas:  21
Index(['Dt_Hr', 'Lat', 'Long', 'Alt', 'Precip', 'Pres_Atm', 'Pres_Atm_max',
       'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv', 'Temp_Amb_max',
       'Temp_Amb_min', 'Pto_Orv_max', 'Pto_Orv_min', 'Umidade_max',
       'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj', 'Vento_vel'],
      dtype='object')
2 :  FORTE_COPACABANA\INMET_SE_RJ_A652_FORTE DE COPACABANA_01-01-2010_A_31-12-2010.CSV linhas:  8760 colunas:  21
Index(['Dt_Hr', 'Lat', 'Long', 'Alt', 'Pre

In [129]:
dataset.head(2)

,Dt_Hr,timestamp,Lat,Long,Alt,Precip,Pres_Atm,Pres_Atm_max,Pres_Atm_min,Rad,...,Temp_Amb_max,Temp_Amb_min,Pto_Orv_max,Pto_Orv_min,Umidade_max,Umidade_min,Umidade,Vento_dir,Vento_raj,Vento_vel
0,2008-01-01 00:00:00,1.199156e+09,-22.988333,-43.190278,42.0,0.0,1006.3,1006.3,1005.7,0.0,...,26.0,25.4,23.3,22.6,88.0,81.0,87.0,257.0,5.3,4.3
1,2008-01-01 01:00:00,1.199160e+09,-22.988333,-43.190278,42.0,0.0,1006.7,1006.8,1006.3,0.0,...,25.7,25.5,23.5,23.2,88.0,87.0,88.0,254.0,5.8,4.1


In [130]:
dataset['Dt_Hr'].min()

Timestamp('2007-05-18 00:00:00')

In [131]:
dataset['Dt_Hr'].max()

Timestamp('2024-11-30 23:00:00')

In [132]:
caminho_arquivo = "FORTE_COPACABANA/CONCATENADO/FORTE_COPACABANA.csv"
dataset.to_csv(caminho_arquivo, index=False, encoding='utf-8')

# MARAMBAIA 

In [134]:
# Listar arquivos

warnings.filterwarnings('ignore')

path = 'MARAMBAIA'

lista_arq = glob.glob(path + "/*.csv")

lista_arq, len(lista_arq)

(['MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2003_A_31-12-2003.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2004_A_31-12-2004.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2005_A_31-12-2005.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2006_A_31-12-2006.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2007_A_31-12-2007.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2008_A_31-12-2008.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2009_A_31-12-2009.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2010_A_31-12-2010.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2011_A_31-12-2011.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2012_A_31-12-2012.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2013_A_31-12-2013.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2014_A_31-12-2014.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2015_A_31-12-2015.CSV',
  'MARAMBAIA\\INMET_SE_RJ_A602_MARAMBAIA_01-01-2016_A_31-12-2016.CSV',
  'MAR

In [135]:
# Gerar dataset e padronizar nomes
nomes = ['Dt_Hr', 'Precip', 'Pres_Atm', 'Pres_Atm_max', 'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv', 'Temp_Amb_max', 'Temp_Amb_min', 'Pto_Orv_max',
        'Pto_Orv_min', 'Umidade_max', 'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj', 'Vento_vel']

local = ['LATITUDE:', 'LONGITUDE:', 'ALTITUDE:']

dataset = concat_dfs(lista_arq, skip=8, colunas=nomes, loc_cols = local, verb=True)

0 :  MARAMBAIA\INMET_SE_RJ_A602_MARAMBAIA_01-01-2003_A_31-12-2003.CSV linhas:  8760 colunas:  21
Index(['Dt_Hr', 'Lat', 'Long', 'Alt', 'Precip', 'Pres_Atm', 'Pres_Atm_max',
       'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv', 'Temp_Amb_max',
       'Temp_Amb_min', 'Pto_Orv_max', 'Pto_Orv_min', 'Umidade_max',
       'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj', 'Vento_vel'],
      dtype='object')
1 :  MARAMBAIA\INMET_SE_RJ_A602_MARAMBAIA_01-01-2004_A_31-12-2004.CSV linhas:  8784 colunas:  21
Index(['Dt_Hr', 'Lat', 'Long', 'Alt', 'Precip', 'Pres_Atm', 'Pres_Atm_max',
       'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv', 'Temp_Amb_max',
       'Temp_Amb_min', 'Pto_Orv_max', 'Pto_Orv_min', 'Umidade_max',
       'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj', 'Vento_vel'],
      dtype='object')
2 :  MARAMBAIA\INMET_SE_RJ_A602_MARAMBAIA_01-01-2005_A_31-12-2005.CSV linhas:  8760 colunas:  21
Index(['Dt_Hr', 'Lat', 'Long', 'Alt', 'Precip', 'Pres_Atm', 'Pres_Atm_max',
       'Pres_Atm_

In [136]:
dataset.head(2)

,Dt_Hr,timestamp,Lat,Long,Alt,Precip,Pres_Atm,Pres_Atm_max,Pres_Atm_min,Rad,...,Temp_Amb_max,Temp_Amb_min,Pto_Orv_max,Pto_Orv_min,Umidade_max,Umidade_min,Umidade,Vento_dir,Vento_raj,Vento_vel
0,2003-01-01 00:00:00,1.041390e+09,-23.05,-43.6,9.7,0.0,1009.7,1009.7,1009.0,-9999.0,...,24.8,24.4,22.8,22.5,89.0,88.0,89.0,267.0,4.0,1.6
1,2003-01-01 01:00:00,1.041394e+09,-23.05,-43.6,9.7,0.0,1009.7,1009.8,1009.7,-9999.0,...,24.5,23.6,22.6,21.7,90.0,88.0,89.0,40.0,2.9,2.0


In [137]:
dataset['Dt_Hr'].min()

Timestamp('2002-11-08 00:00:00')

In [138]:
dataset['Dt_Hr'].max()

Timestamp('2024-11-30 23:00:00')

In [139]:
caminho_arquivo = "MARAMBAIA/CONCATENADO/MARAMBAIA.csv"
dataset.to_csv(caminho_arquivo, index=False, encoding='utf-8')

# JACAREPAGUA

In [141]:
# Listar arquivos

warnings.filterwarnings('ignore')

path = 'JACAREPAGUA'

lista_arq = glob.glob(path + "/*.csv")

lista_arq, len(lista_arq)

(['JACAREPAGUA\\INMET_SE_RJ_A636_RIO DE JANEIRO - JACAREPAGUA_01-01-2018_A_31-12-2018.CSV',
  'JACAREPAGUA\\INMET_SE_RJ_A636_RIO DE JANEIRO - JACAREPAGUA_01-01-2019_A_31-12-2019.CSV',
  'JACAREPAGUA\\INMET_SE_RJ_A636_RIO DE JANEIRO - JACAREPAGUA_01-01-2020_A_31-12-2020.CSV',
  'JACAREPAGUA\\INMET_SE_RJ_A636_RIO DE JANEIRO - JACAREPAGUA_01-01-2021_A_31-12-2021.CSV',
  'JACAREPAGUA\\INMET_SE_RJ_A636_RIO DE JANEIRO - JACAREPAGUA_01-01-2022_A_31-12-2022.CSV',
  'JACAREPAGUA\\INMET_SE_RJ_A636_RIO DE JANEIRO - JACAREPAGUA_01-01-2023_A_31-12-2023.CSV',
  'JACAREPAGUA\\INMET_SE_RJ_A636_RIO DE JANEIRO - JACAREPAGUA_01-01-2024_A_30-11-2024.CSV',
  'JACAREPAGUA\\INMET_SE_RJ_A636_RIO DE JANEIRO - JACAREPAGUA_10-08-2017_A_31-12-2017.CSV'],
 8)

In [142]:
import re
import pandas as pd
from datetime import datetime

def concat_dfs(lista_arq, skip=0, sep=';', dec=',', verb=False, colunas=None, loc_cols=None):
    df = pd.DataFrame()
    
    for arquivo, arq_ind in zip(lista_arq, range(len(lista_arq))):
        enc = detectar_encoding(arquivo)
        
        data = pd.read_csv(arquivo, sep=sep, skiprows=skip, decimal=dec, encoding=enc, parse_dates=[[0,1]], 
                           date_parser=my_date_parser).dropna(axis=1, how='all')

        # Metadados de localização
        if colunas is not None:
            data.columns = colunas
            
        if skip > 0 and loc_cols is not None:
            # Carregar o cabeçalho de metadados
            meta_data = pd.read_csv(arquivo, sep=sep, nrows=skip, encoding=enc, header=None, index_col=0).T

            # Adicionar declarações de impressão para depuração
            print(f"Arquivo: {arquivo}")
            print(f"META_DATA: {meta_data}")
            print(f"LATITUDE: {meta_data[loc_cols[0]][1]}")
            print(f"LONGITUDE: {meta_data[loc_cols[1]][1]}")
            print(f"ALTITUDE: {meta_data[loc_cols[2]][1]}")

            # Formatar dados de localização
            lat_str = re.sub(r'[^-?\d+(\.\d+)?]', '', meta_data[loc_cols[0]][1].replace(',', '.'))
            long_str = re.sub(r'[^-?\d+(\.\d+)?]', '', meta_data[loc_cols[1]][1].replace(',', '.'))
            alt_str = re.sub(r'[^-?\d+(\.\d+)?]', '', meta_data[loc_cols[2]][1].replace(',', '.'))

            print(f"LAT_STR: {lat_str}")
            print(f"LONG_STR: {long_str}")
            print(f"ALT_STR: {alt_str}")

            if not lat_str or not long_str:
                raise ValueError("Valores de localização ausentes ou incorretos.")

            lat = float(lat_str)
            long = float(long_str)
            alt = float(alt_str) if alt_str and alt_str.lower() != 'f' else 0.0

            data.insert(loc=1, column='Lat', value=lat)  
            data.insert(loc=2, column='Long', value=long)
            data.insert(loc=3, column='Alt', value=alt)
        
        if verb:
            print(arq_ind, ': ', arquivo, 'linhas: ', len(data), 'colunas: ', len(data.columns)) 
            print(data.columns)

        df = pd.concat([df, data], ignore_index=True)

    # Coluna de timestamp
    ts = pd.Series(map(datetime.timestamp, df.iloc[:, 0]))
    df.insert(loc=1, column='timestamp', value=ts)
    
    return df

# Exemplo de uso da função
nomes = ['Dt_Hr', 'Precip', 'Pres_Atm', 'Pres_Atm_max', 'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv', 'Temp_Amb_max', 'Temp_Amb_min', 'Pto_Orv_max',
        'Pto_Orv_min', 'Umidade_max', 'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj', 'Vento_vel']
local = ['LATITUDE:', 'LONGITUDE:', 'ALTITUDE:']

dataset = concat_dfs(lista_arq, skip=8, colunas=nomes, loc_cols=local, verb=True)



Arquivo: JACAREPAGUA\INMET_SE_RJ_A636_RIO DE JANEIRO - JACAREPAGUA_01-01-2018_A_31-12-2018.CSV
META_DATA: 0 REGIÃO: UF:                      ESTAÇÃO: CODIGO (WMO):     LATITUDE:  \
1      SE  RJ  RIO DE JANEIRO - JACAREPAGUA          A636  -22,93972221   

0    LONGITUDE: ALTITUDE: DATA DE FUNDAÇÃO (YYYY-MM-DD):  
1  -43,40277777        20                     2017-08-10  
LATITUDE: -22,93972221
LONGITUDE: -43,40277777
ALTITUDE: 20
LAT_STR: -22.93972221
LONG_STR: -43.40277777
ALT_STR: 20
0 :  JACAREPAGUA\INMET_SE_RJ_A636_RIO DE JANEIRO - JACAREPAGUA_01-01-2018_A_31-12-2018.CSV linhas:  8760 colunas:  21
Index(['Dt_Hr', 'Lat', 'Long', 'Alt', 'Precip', 'Pres_Atm', 'Pres_Atm_max',
       'Pres_Atm_min', 'Rad', 'Temp_Amb', 'Pto_Orv', 'Temp_Amb_max',
       'Temp_Amb_min', 'Pto_Orv_max', 'Pto_Orv_min', 'Umidade_max',
       'Umidade_min', 'Umidade', 'Vento_dir', 'Vento_raj', 'Vento_vel'],
      dtype='object')
Arquivo: JACAREPAGUA\INMET_SE_RJ_A636_RIO DE JANEIRO - JACAREPAGUA_01-01-2019_A_31

In [143]:
dataset.head(2)

,Dt_Hr,timestamp,Lat,Long,Alt,Precip,Pres_Atm,Pres_Atm_max,Pres_Atm_min,Rad,...,Temp_Amb_max,Temp_Amb_min,Pto_Orv_max,Pto_Orv_min,Umidade_max,Umidade_min,Umidade,Vento_dir,Vento_raj,Vento_vel
0,2018-01-01 00:00:00,1.514776e+09,-22.939722,-43.402778,20.0,0.0,1008.0,1008.0,1007.5,-9999.0,...,24.1,23.7,20.8,20.5,83.0,81.0,82.0,239.0,1.4,0.1
1,2018-01-01 01:00:00,1.514779e+09,-22.939722,-43.402778,20.0,0.0,1007.9,1008.1,1007.9,-9999.0,...,23.7,23.2,20.6,20.4,84.0,82.0,84.0,44.0,1.2,0.1


In [144]:
dataset['Dt_Hr'].min()

Timestamp('2017-08-10 00:00:00')

In [145]:
dataset['Dt_Hr'].max()

Timestamp('2024-11-30 23:00:00')

In [146]:
caminho_arquivo = "JACAREPAGUA/CONCATENADO/JACAREPAGUA.csv"
dataset.to_csv(caminho_arquivo, index=False, encoding='utf-8')

# JUNTANDO TUDO

In [148]:
import pandas as pd
import os

def adicionar_estacao_e_concatenar(lista_arquivos, pasta_destino, nome_arquivo_final):
    df_final = pd.DataFrame()
    
    for caminho_arquivo in lista_arquivos:
        # Extrair o nome da estação a partir do nome do arquivo sem a extensão .csv
        nome_estacao = os.path.basename(caminho_arquivo).replace('.csv', '')
        
        # Ler o arquivo CSV
        df = pd.read_csv(caminho_arquivo)
        
        # Adicionar a coluna ESTACAO
        df['ESTACAO'] = nome_estacao
        
        # Concatenar os dados ao dataframe final
        df_final = pd.concat([df_final, df], ignore_index=True)
        
    # Salvar o dataframe final em um novo arquivo CSV
    caminho_final = os.path.join(pasta_destino, nome_arquivo_final)
    df_final.to_csv(caminho_final, index=False, sep=';')

# Lista de arquivos CSV
lista_arquivos = [
    "JACAREPAGUA/CONCATENADO/JACAREPAGUA.csv",
    "MARAMBAIA/CONCATENADO/MARAMBAIA.csv",
    "FORTE_COPACABANA/CONCATENADO/FORTE_COPACABANA.csv",
    "VILA_MILITAR/CONCATENADO/VILA_MILITAR.csv"
]

# Pasta de destino e nome do arquivo final
pasta_destino = r"C:\Users\Pichau\INMET\TODAS_ESTACOES"
nome_arquivo_final = "TODAS_ESTACOES_CONCATENADO.csv"

# Chamar a função para adicionar a coluna ESTACAO e concatenar os arquivos
adicionar_estacao_e_concatenar(lista_arquivos, pasta_destino, nome_arquivo_final)


In [149]:
import pandas as pd
import os

def adicionar_estacao_e_concatenar_parquet(lista_arquivos, pasta_destino, nome_arquivo_final):
    df_final = pd.DataFrame()
    
    for caminho_arquivo in lista_arquivos:
        # Extrair o nome da estação a partir do nome do arquivo sem a extensão .csv
        nome_estacao = os.path.basename(caminho_arquivo).replace('.csv', '')
        
        # Ler o arquivo CSV
        df = pd.read_csv(caminho_arquivo)
        
        # Adicionar a coluna ESTACAO
        df['ESTACAO'] = nome_estacao
        
        # Concatenar os dados ao dataframe final
        df_final = pd.concat([df_final, df], ignore_index=True)
        
    # Salvar o dataframe final em um novo arquivo CSV
    caminho_final_csv = os.path.join(pasta_destino, nome_arquivo_final)
    df_final.to_csv(caminho_final_csv, index=False, sep=';')
    
    # Salvar o dataframe final em um novo arquivo Parquet
    caminho_final_parquet = os.path.join(pasta_destino, nome_arquivo_final.replace('.csv', '.parquet'))
    df_final.to_parquet(caminho_final_parquet, index=False)

# Lista de arquivos CSV
lista_arquivos = [
    "JACAREPAGUA/CONCATENADO/JACAREPAGUA.csv",
    "MARAMBAIA/CONCATENADO/MARAMBAIA.csv",
    "FORTE_COPACABANA/CONCATENADO/FORTE_COPACABANA.csv",
    "VILA_MILITAR/CONCATENADO/VILA_MILITAR.csv"
]

# Pasta de destino e nome do arquivo final
pasta_destino = r"C:\Users\Pichau\INMET\TODAS_ESTACOES"
nome_arquivo_final = "TODAS_ESTACOES_CONCATENADO.parquet"

# Chamar a função para adicionar a coluna ESTACAO e concatenar os arquivos
adicionar_estacao_e_concatenar_parquet(lista_arquivos, pasta_destino, nome_arquivo_final)
